In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Employee Data").getOrCreate()

In [ ]:
data = [
    ("Rahul", 60000, "IT"),
    ("Sneha", 45000, "HR"),
    ("Arjun", 70000, "IT"),
    ("Priya", 52000, "Finance"),
    ("Amit", 48000, "HR")
]

columns = ["Name", "Salary", "Department"]

df = spark.createDataFrame(data, columns)

df.show()

+-----+------+----------+
| Name|Salary|Department|
+-----+------+----------+
|Rahul| 60000|        IT|
|Sneha| 45000|        HR|
|Arjun| 70000|        IT|
|Priya| 52000|   Finance|
| Amit| 48000|        HR|
+-----+------+----------+



In [ ]:
df.select("Name", "Salary").show()

+-----+------+
| Name|Salary|
+-----+------+
|Rahul| 60000|
|Sneha| 45000|
|Arjun| 70000|
|Priya| 52000|
| Amit| 48000|
+-----+------+



In [ ]:
df.filter(df.Salary > 50000).show()

+-----+------+----------+
| Name|Salary|Department|
+-----+------+----------+
|Rahul| 60000|        IT|
|Arjun| 70000|        IT|
|Priya| 52000|   Finance|
+-----+------+----------+



In [ ]:
df.groupBy("Department").count().show()

+----------+-----+
|Department|count|
+----------+-----+
|        HR|    2|
|        IT|    2|
|   Finance|    1|
+----------+-----+



In [ ]:
df.orderBy("Salary").show()

+-----+------+----------+
| Name|Salary|Department|
+-----+------+----------+
|Sneha| 45000|        HR|
| Amit| 48000|        HR|
|Priya| 52000|   Finance|
|Rahul| 60000|        IT|
|Arjun| 70000|        IT|
+-----+------+----------+



In [ ]:
df.orderBy(df.Salary.desc()).show()

+-----+------+----------+
| Name|Salary|Department|
+-----+------+----------+
|Arjun| 70000|        IT|
|Rahul| 60000|        IT|
|Priya| 52000|   Finance|
| Amit| 48000|        HR|
|Sneha| 45000|        HR|
+-----+------+----------+



In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_date, year, month, dayofmonth

# Create Spark session
spark = SparkSession.builder.appName("EmployeeExample").getOrCreate()

# Data
data = [
    (1, "Rahul", "IT", 70000, "2024-01-10"),
    (2, "Sneha", "HR", 60000, "2024-02-15"),
    (3, "Arjun", "IT", 75000, "2024-03-01"),
    (4, "Priya", "Finance", 80000, "2024-01-25"),
    (5, "Karan", None, 50000, "2024-02-20")
]

columns = ["emp_id", "name", "department", "salary", "joining_date"]

# Create DataFrame
df = spark.createDataFrame(data, columns)

# Convert joining_date to date type
df_dates = df.withColumn("joining_date", to_date("joining_date"))

df_dates.show()

# Extract year, month, day
df_dates.select(
    "name",
    "joining_date",
    year("joining_date").alias("year"),
    month("joining_date").alias("month"),
    dayofmonth("joining_date").alias("day")
).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
+------+-----+----------+------+------------+

+-----+------------+----+-----+---+
| name|joining_date|year|month|day|
+-----+------------+----+-----+---+
|Rahul|  2024-01-10|2024|    1| 10|
|Sneha|  2024-02-15|2024|    2| 15|
|Arjun|  2024-03-01|2024|    3|  1|
|Priya|  2024-01-25|2024|    1| 25|
|Karan|  2024-02-20|2024|    2| 20|
+-----+------------+----+-----+---+



In [3]:

new_data = [
    (6, "Meera", "IT", 72000, "2024-04-01"),
    (7, "Amit", "HR", 58000, "2024-04-10")
]

new_df = spark.createDataFrame(new_data, columns)
new_df.show()

df.union(new_df).show()
df.unionByName(new_df).show()

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     6|Meera|        IT| 72000|  2024-04-01|
|     7| Amit|        HR| 58000|  2024-04-10|
+------+-----+----------+------+------------+

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|  2024-02-15|
|     3|Arjun|        IT| 75000|  2024-03-01|
|     4|Priya|   Finance| 80000|  2024-01-25|
|     5|Karan|      NULL| 50000|  2024-02-20|
|     6|Meera|        IT| 72000|  2024-04-01|
|     7| Amit|        HR| 58000|  2024-04-10|
+------+-----+----------+------+------------+

+------+-----+----------+------+------------+
|emp_id| name|department|salary|joining_date|
+------+-----+----------+------+------------+
|     1|Rahul|        IT| 70000|  2024-01-10|
|     2|Sneha|        HR| 60000|

In [4]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
def salary_band(salary):
    if salary >= 75000:
        return "High"
    elif salary >= 60000:
        return "Medium"
    else:
        return "Low"
salary_band_udf = udf(salary_band, StringType())
df.withColumn("salary_band", salary_band_udf("salary")).show()#udf

+------+-----+----------+------+------------+-----------+
|emp_id| name|department|salary|joining_date|salary_band|
+------+-----+----------+------+------------+-----------+
|     1|Rahul|        IT| 70000|  2024-01-10|     Medium|
|     2|Sneha|        HR| 60000|  2024-02-15|     Medium|
|     3|Arjun|        IT| 75000|  2024-03-01|       High|
|     4|Priya|   Finance| 80000|  2024-01-25|       High|
|     5|Karan|      NULL| 50000|  2024-02-20|        Low|
+------+-----+----------+------+------------+-----------+

